# Day 1: Exploring the data and building my first tracker

My goal for this notebook is to understand the data and build a complete first solution that I can inspect from beginning to end. It creates a valid `submission.csv`, but the model itself is intentionally simple.

My [visual primer](https://github.com/Caffeinated-Code/biohub-cell-tracking-learning/blob/main/learningNotes/visualPrimer.md) explains the terms used here: [voxel](https://github.com/Caffeinated-Code/biohub-cell-tracking-learning/blob/main/learningNotes/visualPrimer.md#1-the-data-a-3d-world-changing-through-time), [node](https://github.com/Caffeinated-Code/biohub-cell-tracking-learning/blob/main/learningNotes/visualPrimer.md#2-detection-pixels-become-nodes), [sparse labels](https://github.com/Caffeinated-Code/biohub-cell-tracking-learning/blob/main/learningNotes/visualPrimer.md#sparse-labels), [edge](https://github.com/Caffeinated-Code/biohub-cell-tracking-learning/blob/main/learningNotes/visualPrimer.md#3-association-nodes-become-tracks), and [division](https://github.com/Caffeinated-Code/biohub-cell-tracking-learning/blob/main/learningNotes/visualPrimer.md#4-division-a-track-becomes-a-lineage).

> This notebook uses only the official competition data. It does not use external data or model weights. Internet should remain off when the notebook is submitted.

## The entire system in one picture

```text
4D movie (T,Z,Y,X)
        │
        ▼
detect bright, cell-sized 3D blobs ──► nodes
        │
        ▼
link plausible neighbors at t and t+1 ──► edges
        │
        ▼
validate directed graph ──► submission.csv
```

I will inspect each stage before moving to the next one: data → detections → links → submission.

In [ ]:
from pathlib import Path
import json, math, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import blosc2
from scipy.ndimage import gaussian_filter, maximum_filter
from scipy.spatial import cKDTree

warnings.filterwarnings('ignore')
np.random.seed(42)
print('numpy', np.__version__, '| pandas', pd.__version__, '| blosc2', blosc2.__version__)

## 1. Configuration

I keep the main settings in one place so I can see exactly what changes between runs. Distances are measured in microns because `Z`, `Y`, and `X` have different physical spacing. These are starting values rather than tuned parameters.

In [ ]:
COMPETITION_SLUG = 'biohub-cell-tracking-during-development'
SPACING_UM = np.array([1.625, 0.40625, 0.40625], dtype=np.float32)  # Z,Y,X

CONFIG = {
    'small_sigma_um': 1.0,       # preserve cell-sized bright structures
    'large_sigma_um': 2.2,       # estimate slower-varying background
    'nms_radius_um': 2.5,        # keep one peak in a local neighborhood
    'response_quantile': 0.995,  # adaptive per-frame candidate threshold
    'max_nodes_per_frame': 1800, # limit memory use and excess detections
    'max_link_um': 7.0,          # maximum movement between adjacent frames
}
RUN_FULL_SUBMISSION = True
print(json.dumps(CONFIG, indent=2))

In [ ]:
def find_competition_root():
    candidates = [
        Path('/kaggle/input/competitions') / COMPETITION_SLUG,
        Path('/kaggle/input') / COMPETITION_SLUG,
    ]
    for path in candidates:
        if (path / 'train').exists() and (path / 'test').exists():
            return path
    matches = [p.parent for p in Path('/kaggle/input').rglob('train')
               if p.is_dir() and (p.parent / 'test').exists()]
    if matches:
        return matches[0]
    raise FileNotFoundError('Attach the official competition dataset to this notebook.')

ROOT = find_competition_root()
TRAIN_DIR, TEST_DIR = ROOT / 'train', ROOT / 'test'
train_zarrs = sorted(TRAIN_DIR.glob('*.zarr'))
test_zarrs = sorted(TEST_DIR.glob('*.zarr'))
sample_candidates = list(ROOT.glob('*sample*submission*.csv'))

print('Competition root:', ROOT)
print('Train movies:', len(train_zarrs), '| Test movies:', len(test_zarrs))
print('Train GEFF graphs:', len(list(TRAIN_DIR.glob('*.geff'))))
print('Sample submission:', sample_candidates[0] if sample_candidates else 'not found (not required)')
assert train_zarrs and test_zarrs, 'Expected both train and test OME-Zarr movies.'

## 2. Explore the images

I start by checking the movie shapes, data types, chunks, and intensity ranges. The competition stores one timepoint per compressed Zarr v3 chunk, so the small reader above loads one frame at a time without adding a new package. I then view the volume from three directions. A maximum-intensity projection is useful for orientation, but it hides depth, so the detector still works on the full 3D image.

In [ ]:
class TimeChunkedZarr:
    """Small reader for this competition's time-chunked Zarr v3 arrays."""
    def __init__(self, path):
        self.path = Path(path) / '0'
        metadata_path = self.path / 'zarr.json'
        if not metadata_path.exists():
            raise FileNotFoundError(f'Missing Zarr metadata: {metadata_path}')
        self.metadata = json.loads(metadata_path.read_text())
        self.shape = tuple(self.metadata['shape'])
        self.chunks = tuple(self.metadata['chunk_grid']['configuration']['chunk_shape'])
        self.dtype = np.dtype(self.metadata['data_type'])
        expected = (1,) + self.shape[1:]
        if self.chunks != expected:
            raise ValueError(f'Expected one full frame per chunk, found {self.chunks}')

    def __len__(self):
        return self.shape[0]

    def __getitem__(self, t):
        if not isinstance(t, (int, np.integer)):
            raise TypeError('This reader loads one integer timepoint at a time.')
        t = int(t) % self.shape[0]
        chunk_path = self.path / 'c' / str(t) / '0' / '0' / '0'
        if not chunk_path.exists():
            return np.zeros(self.shape[1:], dtype=self.dtype)
        raw = blosc2.decompress(chunk_path.read_bytes())
        return np.frombuffer(raw, dtype=self.dtype).reshape(self.chunks)[0]

def open_movie(path):
    arr = TimeChunkedZarr(path)
    return arr.metadata, arr

audit_rows = []
for split, paths in [('train', train_zarrs), ('test', test_zarrs)]:
    for path in paths:
        group, arr = open_movie(path)
        audit_rows.append({
            'split': split, 'dataset': path.stem, 'shape': tuple(arr.shape),
            'dtype': str(arr.dtype), 'chunks': tuple(arr.chunks),
            'frames': int(arr.shape[0]),
        })
audit = pd.DataFrame(audit_rows)
display(audit)
display(audit.groupby('split')['frames'].agg(['count','min','median','max','sum']))

In [ ]:
eda_path = train_zarrs[0]
eda_group, eda_arr = open_movie(eda_path)
eda_t = len(eda_arr) // 2
volume = np.asarray(eda_arr[eda_t], dtype=np.float32)
sampled = volume[::max(1, volume.shape[0]//32), ::4, ::4]
q = np.quantile(sampled, [0, .01, .5, .95, .99, .999, 1])
print('Movie:', eda_path.stem, '| frame:', eda_t, '| volume shape:', volume.shape)
print('Sampled intensity quantiles:', dict(zip([0,.01,.5,.95,.99,.999,1], q.round(2))))

z0, y0, x0 = np.unravel_index(np.argmax(volume), volume.shape)
vmin, vmax = np.quantile(sampled, [0.50, 0.999])
fig, ax = plt.subplots(1, 3, figsize=(16, 5))
ax[0].imshow(volume.max(axis=0), cmap='gray', vmin=vmin, vmax=vmax); ax[0].set_title('XY maximum projection')
ax[1].imshow(volume.max(axis=1), cmap='gray', vmin=vmin, vmax=vmax); ax[1].set_title('XZ maximum projection')
ax[2].imshow(volume.max(axis=2), cmap='gray', vmin=vmin, vmax=vmax); ax[2].set_title('YZ maximum projection')
for a in ax: a.axis('off')
plt.suptitle(f'{eda_path.stem}, t={eda_t} — projections are for orientation, not detection')
plt.tight_layout(); plt.show()

## 3. Detect candidate nodes in 3D

I use [Difference of Gaussians](https://github.com/Caffeinated-Code/biohub-cell-tracking-learning/blob/main/learningNotes/visualPrimer.md#day-1-detector-difference-of-gaussians) followed by [non-maximum suppression](https://github.com/Caffeinated-Code/biohub-cell-tracking-learning/blob/main/learningNotes/visualPrimer.md#day-1-detector-difference-of-gaussians). Sigma and suppression sizes are converted from microns to voxels. This matters because a raw one-voxel move has different physical meaning along `Z` and `X/Y`.

In [ ]:
def odd_window(radius_um):
    radii = np.maximum(1, np.ceil(radius_um / SPACING_UM).astype(int))
    return tuple((2 * radii + 1).tolist())

def detect_nodes(frame, config=CONFIG):
    frame = np.asarray(frame, dtype=np.float32)
    sample = frame[::max(1, frame.shape[0]//32), ::4, ::4]
    lo, hi = np.quantile(sample, [0.01, 0.9995])
    image = np.clip((frame - lo) / max(hi - lo, 1.0), 0, 1)

    sigma_small = tuple((config['small_sigma_um'] / SPACING_UM).tolist())
    sigma_large = tuple((config['large_sigma_um'] / SPACING_UM).tolist())
    response = gaussian_filter(image, sigma_small) - gaussian_filter(image, sigma_large)
    positive = response[response > 0]
    if positive.size == 0:
        return np.empty((0, 3), dtype=np.int32), np.empty(0, dtype=np.float32)

    threshold = np.quantile(positive, config['response_quantile'])
    local_max = response == maximum_filter(response, size=odd_window(config['nms_radius_um']), mode='nearest')
    coords = np.argwhere(local_max & (response >= threshold))
    scores = response[tuple(coords.T)] if len(coords) else np.empty(0, dtype=np.float32)

    cap = config['max_nodes_per_frame']
    if len(coords) > cap:
        keep = np.argpartition(scores, -cap)[-cap:]
        coords, scores = coords[keep], scores[keep]
    order = np.argsort(scores)[::-1]
    return coords[order].astype(np.int32), scores[order].astype(np.float32)

demo_coords, demo_scores = detect_nodes(volume)
print('Detected candidates:', len(demo_coords))
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(volume.max(axis=0), cmap='gray', vmin=vmin, vmax=vmax)
if len(demo_coords):
    ax.scatter(demo_coords[:,2], demo_coords[:,1], s=10, facecolors='none', edgecolors='cyan', linewidths=.6)
ax.set_title('3D detections projected onto XY (cyan circles)'); ax.axis('off'); plt.show()

## 4. Link adjacent timepoints

My first tracker considers pairs that are close enough in physical space. It checks the shortest possible links first and uses each node at most once. This is easy to inspect, but it does not handle divisions, crossings, changes in appearance, or missed detections.

In [ ]:
def greedy_links(prev_coords, prev_ids, curr_coords, curr_ids, max_distance_um):
    if len(prev_coords) == 0 or len(curr_coords) == 0:
        return []
    prev_um = prev_coords * SPACING_UM
    curr_um = curr_coords * SPACING_UM
    tree = cKDTree(curr_um)
    candidates = []
    for i, neighbors in enumerate(tree.query_ball_point(prev_um, r=max_distance_um)):
        for j in neighbors:
            distance = float(np.linalg.norm(prev_um[i] - curr_um[j]))
            candidates.append((distance, i, j))
    candidates.sort()
    used_prev, used_curr, links = set(), set(), []
    for distance, i, j in candidates:
        if i not in used_prev and j not in used_curr:
            links.append((int(prev_ids[i]), int(curr_ids[j]), distance))
            used_prev.add(i); used_curr.add(j)
    return links

def process_movie(path, config=CONFIG, verbose=True):
    _, arr = open_movie(path)
    node_rows, edge_rows = [], []
    next_node_id = 1
    prev_coords = np.empty((0, 3), dtype=np.int32)
    prev_ids = np.empty(0, dtype=np.int64)
    start = time.time()

    for t in range(arr.shape[0]):
        coords, scores = detect_nodes(arr[t], config)
        ids = np.arange(next_node_id, next_node_id + len(coords), dtype=np.int64)
        next_node_id += len(coords)
        for node_id, (z, y, x), score in zip(ids, coords, scores):
            node_rows.append((path.stem, 'node', node_id, t, int(z), int(y), int(x), -1, -1))
        for source, target, distance in greedy_links(prev_coords, prev_ids, coords, ids, config['max_link_um']):
            edge_rows.append((path.stem, 'edge', -1, -1, -1, -1, -1, source, target))
        prev_coords, prev_ids = coords, ids
        if verbose and (t == 0 or (t + 1) % 10 == 0 or t + 1 == arr.shape[0]):
            print(f'  {path.stem}: frame {t+1:>3}/{arr.shape[0]} | nodes={len(node_rows):>7} | edges={len(edge_rows):>7}')

    if verbose:
        print(f'  finished {path.stem} in {(time.time()-start)/60:.1f} min')
    return node_rows + edge_rows

# Tiny logic check independent of competition data.
p = np.array([[0,0,0], [0,20,20]])
c = np.array([[0,1,1], [0,21,20]])
assert len(greedy_links(p, np.array([1,2]), c, np.array([3,4]), 7.0)) == 2
print('Tracker unit check passed.')

## 5. Run test inference and create the submission

The competition requires exactly two row types: node rows containing coordinates and edge rows containing source/target node IDs. The final checks catch duplicate IDs, dangling edges, backward-time edges, and accidental merges before I submit the notebook.

In [ ]:
COLUMNS = ['dataset','row_type','node_id','t','z','y','x','source_id','target_id']
all_rows = []
paths_to_run = test_zarrs if RUN_FULL_SUBMISSION else test_zarrs[:1]
for movie_path in paths_to_run:
    print(f'Processing {movie_path.stem} ...')
    all_rows.extend(process_movie(movie_path))

submission = pd.DataFrame(all_rows, columns=COLUMNS)
submission.insert(0, 'id', np.arange(len(submission), dtype=np.int64))
integer_columns = ['id','node_id','t','z','y','x','source_id','target_id']
submission[integer_columns] = submission[integer_columns].astype(np.int64)
display(submission.head())
display(submission.groupby(['dataset','row_type']).size().unstack(fill_value=0))

In [ ]:
def validate_submission(df, expected_datasets):
    expected_columns = ['id'] + COLUMNS
    assert list(df.columns) == expected_columns
    assert not df.empty and df['id'].is_unique
    assert set(df.row_type) <= {'node','edge'}
    assert set(df.dataset) == set(expected_datasets)

    for dataset, part in df.groupby('dataset'):
        nodes = part[part.row_type == 'node']
        edges = part[part.row_type == 'edge']
        assert len(nodes) > 0, f'{dataset}: no nodes'
        assert nodes.node_id.is_unique, f'{dataset}: duplicate node IDs'
        node_times = dict(zip(nodes.node_id, nodes.t))
        node_ids = set(node_times)
        assert set(edges.source_id) <= node_ids, f'{dataset}: dangling sources'
        assert set(edges.target_id) <= node_ids, f'{dataset}: dangling targets'
        if len(edges):
            assert edges.source_id.value_counts().max() <= 1, f'{dataset}: unexpected fork'
            assert edges.target_id.value_counts().max() <= 1, f'{dataset}: merge detected'
            assert all(node_times[tgt] == node_times[src] + 1 for src, tgt in zip(edges.source_id, edges.target_id))
    return True

expected = [p.stem for p in paths_to_run]
assert validate_submission(submission, expected)
OUTPUT = Path('/kaggle/working/submission.csv')
submission.to_csv(OUTPUT, index=False)
print(f'Validated and wrote {len(submission):,} rows to {OUTPUT}')
print('File size:', round(OUTPUT.stat().st_size / 2**20, 2), 'MiB')

## 6. What this first pass gives me

At this point I have a complete path from the microscopy movies to a submission file. I have also checked the coordinate convention, physical-distance calculation, graph structure, and CSV format.

This is only a starting point. Difference of Gaussians does not learn cell appearance, and the tracker only uses distance between adjacent frames. It does not repair gaps or predict cell divisions.

When this notebook finishes, I check the node and edge counts for every test movie. I then use **Save Version → Save & Run All**, confirm that internet is off, and submit the generated `submission.csv`.